In [1]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Data

In [5]:
res = []
resource = "planets"
query = f"https://swapi.tech/api/{resource}"
json_request = requests.get(query).json()
res.extend(json_request["results"])

while json_request["next"]:
    json_request = requests.get(json_request["next"]).json()
    res.extend(json_request["results"])

In [6]:
res

[{'uid': '1',
  'name': 'Tatooine',
  'url': 'https://www.swapi.tech/api/planets/1'},
 {'uid': '2',
  'name': 'Alderaan',
  'url': 'https://www.swapi.tech/api/planets/2'},
 {'uid': '3',
  'name': 'Yavin IV',
  'url': 'https://www.swapi.tech/api/planets/3'},
 {'uid': '4', 'name': 'Hoth', 'url': 'https://www.swapi.tech/api/planets/4'},
 {'uid': '5',
  'name': 'Dagobah',
  'url': 'https://www.swapi.tech/api/planets/5'},
 {'uid': '6', 'name': 'Bespin', 'url': 'https://www.swapi.tech/api/planets/6'},
 {'uid': '7', 'name': 'Endor', 'url': 'https://www.swapi.tech/api/planets/7'},
 {'uid': '8', 'name': 'Naboo', 'url': 'https://www.swapi.tech/api/planets/8'},
 {'uid': '9',
  'name': 'Coruscant',
  'url': 'https://www.swapi.tech/api/planets/9'},
 {'uid': '10',
  'name': 'Kamino',
  'url': 'https://www.swapi.tech/api/planets/10'},
 {'uid': '11',
  'name': 'Geonosis',
  'url': 'https://www.swapi.tech/api/planets/11'},
 {'uid': '12',
  'name': 'Utapau',
  'url': 'https://www.swapi.tech/api/planets/

In [7]:
df = spark.createDataFrame(res)
df.show()

+--------------+---+--------------------+
|          name|uid|                 url|
+--------------+---+--------------------+
|      Tatooine|  1|https://www.swapi...|
|      Alderaan|  2|https://www.swapi...|
|      Yavin IV|  3|https://www.swapi...|
|          Hoth|  4|https://www.swapi...|
|       Dagobah|  5|https://www.swapi...|
|        Bespin|  6|https://www.swapi...|
|         Endor|  7|https://www.swapi...|
|         Naboo|  8|https://www.swapi...|
|     Coruscant|  9|https://www.swapi...|
|        Kamino| 10|https://www.swapi...|
|      Geonosis| 11|https://www.swapi...|
|        Utapau| 12|https://www.swapi...|
|      Mustafar| 13|https://www.swapi...|
|      Kashyyyk| 14|https://www.swapi...|
|   Polis Massa| 15|https://www.swapi...|
|       Mygeeto| 16|https://www.swapi...|
|       Felucia| 17|https://www.swapi...|
|Cato Neimoidia| 18|https://www.swapi...|
|     Saleucami| 19|https://www.swapi...|
|       Stewjon| 20|https://www.swapi...|
+--------------+---+--------------

In [8]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [9]:
df = df.withColumn("properties", get_properties(F.col("url")))

In [10]:
df.show()

+--------------+---+--------------------+--------------------+
|          name|uid|                 url|          properties|
+--------------+---+--------------------+--------------------+
|      Tatooine|  1|https://www.swapi...|{"created": "2025...|
|      Alderaan|  2|https://www.swapi...|{"created": "2025...|
|      Yavin IV|  3|https://www.swapi...|{"created": "2025...|
|          Hoth|  4|https://www.swapi...|{"created": "2025...|
|       Dagobah|  5|https://www.swapi...|{"created": "2025...|
|        Bespin|  6|https://www.swapi...|{"created": "2025...|
|         Endor|  7|https://www.swapi...|{"created": "2025...|
|         Naboo|  8|https://www.swapi...|{"created": "2025...|
|     Coruscant|  9|https://www.swapi...|{"created": "2025...|
|        Kamino| 10|https://www.swapi...|{"created": "2025...|
|      Geonosis| 11|https://www.swapi...|{"created": "2025...|
|        Utapau| 12|https://www.swapi...|{"created": "2025...|
|      Mustafar| 13|https://www.swapi...|{"created": "2

In [11]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

DataFrame[]

In [12]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Overwrite

In [13]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [14]:
class StarWarsBronze2(bronze.Bronze):
    def custom_load(self, table):
        df = spark.range(5).withColumn("t", F.lit(table))
        return df


bronze_instance2 = StarWarsBronze2(spark, **options)

In [15]:
# Ensure the schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

# Run one table
bronze_instance.load().transform().write(mode="overwrite").execute("people")

2025-03-15 23:37:59 | people | execute | Started
2025-03-15 23:37:59 | people | execute | Started
2025-03-15 23:37:59 | people | load | Started
2025-03-15 23:38:04 | people | load | Completed in 0.07 min
2025-03-15 23:38:04 | people | transform | Started
2025-03-15 23:38:04 | people | transform | Completed in 0.0 min
2025-03-15 23:38:04 | people | write | Started
2025-03-15 23:38:28 | people | write | Completed in 0.38 min
2025-03-15 23:38:28 | people | execute | Completed in 0.47 min
2025-03-15 23:38:28 | people | execute | Completed in 0.47 min


In [16]:
# run multiple tables
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-15 23:38:28 | people | execute | Started
2025-03-15 23:38:28 | people | execute | Started
2025-03-15 23:38:28 | people | load | Started
2025-03-15 23:38:32 | people | load | Completed in 0.07 min
2025-03-15 23:38:32 | people | transform | Started
2025-03-15 23:38:32 | people | transform | Completed in 0.0 min
2025-03-15 23:38:32 | people | write | Started
2025-03-15 23:38:50 | people | write | Completed in 0.3 min
2025-03-15 23:38:50 | people | execute | Completed in 0.37 min
2025-03-15 23:38:50 | planets | execute | Started
2025-03-15 23:38:50 | planets | load | Started
2025-03-15 23:38:53 | planets | load | Completed in 0.05 min
2025-03-15 23:38:53 | planets | transform | Started
2025-03-15 23:38:53 | planets | transform | Completed in 0.0 min
2025-03-15 23:38:53 | planets | write | Started
2025-03-15 23:39:07 | planets | write | Completed in 0.22 min
2025-03-15 23:39:07 | planets | execute | Completed in 0.27 min
2025-03-15 23:39:07 | people | execute | Completed in 0.65 min

In [17]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show(100)

No. Rows: 82
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-15 23:38:...|         Cliegg Lars| 62|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|   Poggle the Lesser| 63|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|     Luminara Unduli| 64|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|       Barriss Offee| 65|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|               Dormé| 66|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|               Dooku| 67|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...| Bail Prestor Organa| 68|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|          Jango Fett| 69|https://www.swapi...|{"created": "2025..

In [18]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(100)

No. Rows: 60
+--------------------+--------------+---+--------------------+--------------------+
|         LH_BronzeTS|          name|uid|                 url|          properties|
+--------------------+--------------+---+--------------------+--------------------+
|2025-03-15 23:38:...|      Tatooine|  1|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|      Alderaan|  2|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|      Yavin IV|  3|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|          Hoth|  4|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|       Dagobah|  5|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|        Bespin|  6|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|         Endor|  7|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|         Naboo|  8|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:38:...|     Coruscant|  9|https://www.swapi...|{

In [19]:
bronze_instance.data["people"].show()

+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-15 23:39:...|      Luke Skywalker|  1|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|               C-3PO|  2|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|               R2-D2|  3|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|         Darth Vader|  4|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|         Leia Organa|  5|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|           Owen Lars|  6|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|  Beru Whitesun lars|  7|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|               R5-D4|  8|https://www.swapi...|{"created": "2025...|
|2025-03-1

In [20]:
bronze_instance.data["planets"].show()

+--------------------+--------------+---+--------------------+--------------------+
|         LH_BronzeTS|          name|uid|                 url|          properties|
+--------------------+--------------+---+--------------------+--------------------+
|2025-03-15 23:39:...|      Tatooine|  1|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|      Alderaan|  2|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|      Yavin IV|  3|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|          Hoth|  4|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|       Dagobah|  5|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|        Bespin|  6|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|         Endor|  7|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|         Naboo|  8|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|     Coruscant|  9|https://www.swapi...|{"created": "2

# 3 Replace Where

In [21]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))

    def get_replace_condition(self, df: DataFrame, table: str) -> str:
        return "uid > '0'"


bronze_instance = StarWarsBronze(spark, **options)
bronze_instance.load().transform().write(mode="replace").execute("people")

2025-03-15 23:39:44 | people | execute | Started
2025-03-15 23:39:44 | people | execute | Started
2025-03-15 23:39:44 | people | load | Started
2025-03-15 23:39:48 | people | load | Completed in 0.07 min
2025-03-15 23:39:48 | people | transform | Started
2025-03-15 23:39:48 | people | transform | Completed in 0.0 min
2025-03-15 23:39:48 | people | write | Started
2025-03-15 23:40:07 | people | write | Completed in 0.32 min
2025-03-15 23:40:07 | people | execute | Completed in 0.38 min
2025-03-15 23:40:07 | people | execute | Completed in 0.38 min


In [22]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-03-15 23:39:...|        Cliegg Lars| 62|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|  Poggle the Lesser| 63|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|    Luminara Unduli| 64|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|      Barriss Offee| 65|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|              Dormé| 66|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|              Dooku| 67|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|Bail Prestor Organa| 68|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|         Jango Fett| 69|https://www.swapi...|{"created": "2025...|
|2025-03

# 4 Append

In [23]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)
bronze_instance.load().transform().write().execute("people")  # default mode is append

2025-03-15 23:40:08 | people | execute | Started
2025-03-15 23:40:08 | people | execute | Started
2025-03-15 23:40:08 | people | load | Started
2025-03-15 23:40:13 | people | load | Completed in 0.07 min
2025-03-15 23:40:13 | people | transform | Started
2025-03-15 23:40:13 | people | transform | Completed in 0.0 min
2025-03-15 23:40:13 | people | write | Started
2025-03-15 23:40:31 | people | write | Completed in 0.3 min
2025-03-15 23:40:31 | people | execute | Completed in 0.37 min
2025-03-15 23:40:31 | people | execute | Completed in 0.37 min


In [24]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 164
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-15 23:39:...|      Luke Skywalker|  1|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|               C-3PO|  2|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|               R2-D2|  3|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|         Darth Vader|  4|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|         Leia Organa|  5|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|           Owen Lars|  6|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|  Beru Whitesun lars|  7|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:39:...|               R5-D4|  8|https://www.swapi...|{"created": "2025.

# 5 Merge

In [25]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))

    def get_delta_merge_builder(
        self, df: DataFrame, delta_table: DeltaTable
    ) -> DeltaMergeBuilder:
        merge_condition = "target.uid = source.uid"
        builder = delta_table.alias("target").merge(df.alias("source"), merge_condition)
        builder = builder.whenMatchedUpdateAll()
        builder = builder.whenNotMatchedInsertAll()
        return builder


bronze_instance = StarWarsBronze(spark, **options)
bronze_instance.load().transform().write(mode="merge").execute("people")

2025-03-15 23:40:32 | people | execute | Started
2025-03-15 23:40:32 | people | execute | Started
2025-03-15 23:40:32 | people | load | Started
2025-03-15 23:40:36 | people | load | Completed in 0.07 min
2025-03-15 23:40:36 | people | transform | Started
2025-03-15 23:40:36 | people | transform | Completed in 0.0 min
2025-03-15 23:40:36 | people | write | Started
2025-03-15 23:40:55 | people | write | Completed in 0.3 min
2025-03-15 23:40:55 | people | execute | Completed in 0.38 min
2025-03-15 23:40:55 | people | execute | Completed in 0.38 min


In [26]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 164
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-15 23:40:...|      Luke Skywalker|  1|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:40:...|      Luke Skywalker|  1|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:40:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:40:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:40:...|    Anakin Skywalker| 11|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:40:...|    Anakin Skywalker| 11|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:40:...|      Wilhuff Tarkin| 12|https://www.swapi...|{"created": "2025...|
|2025-03-15 23:40:...|      Wilhuff Tarkin| 12|https://www.swapi...|{"created": "2025.

# 6 Clean Up

In [ ]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.stop()

DataFrame[]